In [ ]:

import gc
import math
import os
import random
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import EditedNearestNeighbours


SEEDS = [11, 22, 33, 44, 55]
PARTITION_SEED = 2026

LABEL_COLUMN = "label"
BINARY_COLUMN = "binary_label"
SOURCE_COLUMN = "source"
ORIGINAL_LABEL_COLUMN = "original_label"

MAX_TRAIN_ROWS = None
MAX_TEST_ROWS = None

KAMAL_MAX_TRAIN_ROWS = 175_368
KAMAL_USE_LOF = True
KAMAL_USE_ADASYN_SMOTE_ENN = True

USE_EARLY_STOPPING = True

EPOCHS = {
    "kilichev_1dcnn": 99,
    "buhl_dcnn": 30,
    "kamal_transformer_dnn": 500,
    "elmasry_dpso_dnn": 15,
}

BATCH_SIZES = {
    "kilichev_1dcnn": 64,
    "buhl_dcnn": 128,
    "kamal_transformer_dnn": 128,
    "elmasry_dpso_dnn": 450,
}

MODEL_KEYS = [
    "kilichev_1dcnn",
    "buhl_dcnn",
    "kamal_transformer_dnn",
    "zkr_random_forest",
    "elmasry_dpso_dnn",
]

RESULTS_DIR = Path(
    "/content/drive/MyDrive/Array_submission/Results/CIC_IDS2017/"
    "Experiment4_Model_Specific_Test_Ratios"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Results directory:", RESULTS_DIR)

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Results directory: /content/drive/MyDrive/Array_submission/Results/CIC_IDS2017/Experiment4_Model_Specific_Test_Ratios


In [ ]:

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def binary_y(frame):
    return frame[BINARY_COLUMN].to_numpy(dtype=np.int8)


def optional_sample(frame, maximum_rows, seed):
    if maximum_rows is None or len(frame) <= maximum_rows:
        return frame.reset_index(drop=True).copy()

    sampled, _ = train_test_split(
        frame,
        train_size=maximum_rows,
        random_state=seed,
        shuffle=True,
        stratify=frame[BINARY_COLUMN],
    )
    return sampled.reset_index(drop=True)


def make_callbacks(include_reduce_lr=True):
    if not USE_EARLY_STOPPING:
        return []

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            min_delta=1e-4,
            restore_best_weights=True,
            verbose=1,
        ),
    ]

    if include_reduce_lr:
        callbacks.append(
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1,
            )
        )

    return callbacks

In [ ]:

NATIVE_FILE_PATHS = {
    "Monday-WorkingHours": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Monday-WorkingHours.pcap_ISCX.csv"
    ),
    "Tuesday-WorkingHours": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Tuesday-WorkingHours.pcap_ISCX.csv"
    ),
    "Wednesday-WorkingHours": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Wednesday-workingHours.pcap_ISCX.csv"
    ),
    "Thursday-Morning-WebAttacks": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
    ),
    "Thursday-Afternoon-Infiltration": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv"
    ),
    "Friday-Morning": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Friday-WorkingHours-Morning.pcap_ISCX.csv"
    ),
    "Friday-Afternoon-PortScan": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
    ),
    "Friday-Afternoon-DDoS": (
        "/content/drive/MyDrive/Colab Notebooks/DATASETS/"
        "CIC-IDS-2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
    ),
}

native_parts = []

for source_file, path in NATIVE_FILE_PATHS.items():
    part = pd.read_csv(path)
    part["source_file"] = source_file
    native_parts.append(part)

original_raw = pd.concat(
    native_parts,
    ignore_index=True,
)

del native_parts
gc.collect()

print("Native CIC-IDS2017 raw shape:", original_raw.shape)
print(original_raw["source_file"].value_counts())

Native CIC-IDS2017 raw shape: (2830743, 80)
source_file
Wednesday-WorkingHours             692703
Monday-WorkingHours                529918
Tuesday-WorkingHours               445909
Thursday-Afternoon-Infiltration    288602
Friday-Afternoon-PortScan          286467
Friday-Afternoon-DDoS              225745
Friday-Morning                     191033
Thursday-Morning-WebAttacks        170366
Name: count, dtype: int64


In [ ]:

darpa_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/Darpa/Darpa_Pcap_CIC_features.csv"
)
unsw_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/UNSW-NB15/UNSW_Pcap_CIC_features.csv"
)
cic17_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/CIC2017/CIC17_Pcap_CIC_features.csv"
)
cic18_raw = pd.read_csv(
    "/content/drive/MyDrive/Array_submission/Dataset/"
    "CIC-features/CIC2018/CIC18_Pcap_CIC_features.csv"
)

print("DARPA:", darpa_raw.shape)
print("UNSW-NB15:", unsw_raw.shape)
print("CIC-IDS2017 PCAP:", cic17_raw.shape)
print("CSE-CIC-IDS2018 PCAP:", cic18_raw.shape)

/tmp/ipykernel_3340/2947730386.py:15: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77) have mixed types. Specify dtype option on import or set low_memory=False.
  cic18_raw = pd.read_csv(


DARPA: (2439149, 95)
UNSW-NB15: (313636, 84)
CIC-IDS2017 PCAP: (1233126, 94)
CSE-CIC-IDS2018 PCAP: (1661679, 79)


In [6]:
CIC2017_FEATURE_COLUMNS = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Std",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Total",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "Packet Length Variance",
    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "CWE Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Average Packet Size",
    "Avg Fwd Segment Size",
    "Avg Bwd Segment Size",
    "Fwd Header Length.1",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
    "Subflow Fwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Packets",
    "Subflow Bwd Bytes",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "act_data_pkt_fwd",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Mean",
    "Idle Std",
    "Idle Max",
    "Idle Min",
]

CICFLOWMETER_TO_CIC2017 = {
    "Dst Port": "Destination Port",
    "Total Fwd Packet": "Total Fwd Packets",
    "Total Bwd packets": "Total Backward Packets",
    "Total Length of Fwd Packet": "Total Length of Fwd Packets",
    "Total Length of Bwd Packet": "Total Length of Bwd Packets",
    "Packet Length Min": "Min Packet Length",
    "Packet Length Max": "Max Packet Length",
    "CWR Flag Count": "CWE Flag Count",
    "Fwd Segment Size Avg": "Avg Fwd Segment Size",
    "Bwd Segment Size Avg": "Avg Bwd Segment Size",
    "Fwd Bytes/Bulk Avg": "Fwd Avg Bytes/Bulk",
    "Fwd Packet/Bulk Avg": "Fwd Avg Packets/Bulk",
    "Fwd Bulk Rate Avg": "Fwd Avg Bulk Rate",
    "Bwd Bytes/Bulk Avg": "Bwd Avg Bytes/Bulk",
    "Bwd Packet/Bulk Avg": "Bwd Avg Packets/Bulk",
    "Bwd Bulk Rate Avg": "Bwd Avg Bulk Rate",
    "FWD Init Win Bytes": "Init_Win_bytes_forward",
    "Bwd Init Win Bytes": "Init_Win_bytes_backward",
    "Fwd Act Data Pkts": "act_data_pkt_fwd",
    "Fwd Seg Size Min": "min_seg_size_forward",
}

UNSW_TO_CIC2017 = {
    "dst_port": "Destination Port",
    "flow_duration": "Flow Duration",
    "tot_fwd_pkts": "Total Fwd Packets",
    "tot_bwd_pkts": "Total Backward Packets",
    "totlen_fwd_pkts": "Total Length of Fwd Packets",
    "totlen_bwd_pkts": "Total Length of Bwd Packets",
    "fwd_pkt_len_max": "Fwd Packet Length Max",
    "fwd_pkt_len_min": "Fwd Packet Length Min",
    "fwd_pkt_len_mean": "Fwd Packet Length Mean",
    "fwd_pkt_len_std": "Fwd Packet Length Std",
    "bwd_pkt_len_max": "Bwd Packet Length Max",
    "bwd_pkt_len_min": "Bwd Packet Length Min",
    "bwd_pkt_len_mean": "Bwd Packet Length Mean",
    "bwd_pkt_len_std": "Bwd Packet Length Std",
    "flow_byts_s": "Flow Bytes/s",
    "flow_pkts_s": "Flow Packets/s",
    "flow_iat_mean": "Flow IAT Mean",
    "flow_iat_std": "Flow IAT Std",
    "flow_iat_max": "Flow IAT Max",
    "flow_iat_min": "Flow IAT Min",
    "fwd_iat_tot": "Fwd IAT Total",
    "fwd_iat_mean": "Fwd IAT Mean",
    "fwd_iat_std": "Fwd IAT Std",
    "fwd_iat_max": "Fwd IAT Max",
    "fwd_iat_min": "Fwd IAT Min",
    "bwd_iat_tot": "Bwd IAT Total",
    "bwd_iat_mean": "Bwd IAT Mean",
    "bwd_iat_std": "Bwd IAT Std",
    "bwd_iat_max": "Bwd IAT Max",
    "bwd_iat_min": "Bwd IAT Min",
    "fwd_psh_flags": "Fwd PSH Flags",
    "bwd_psh_flags": "Bwd PSH Flags",
    "fwd_urg_flags": "Fwd URG Flags",
    "bwd_urg_flags": "Bwd URG Flags",
    "fwd_header_len": "Fwd Header Length",
    "bwd_header_len": "Bwd Header Length",
    "fwd_pkts_s": "Fwd Packets/s",
    "bwd_pkts_s": "Bwd Packets/s",
    "pkt_len_min": "Min Packet Length",
    "pkt_len_max": "Max Packet Length",
    "pkt_len_mean": "Packet Length Mean",
    "pkt_len_std": "Packet Length Std",
    "pkt_len_var": "Packet Length Variance",
    "fin_flag_cnt": "FIN Flag Count",
    "syn_flag_cnt": "SYN Flag Count",
    "rst_flag_cnt": "RST Flag Count",
    "psh_flag_cnt": "PSH Flag Count",
    "ack_flag_cnt": "ACK Flag Count",
    "urg_flag_cnt": "URG Flag Count",
    "cwr_flag_count": "CWE Flag Count",
    "ece_flag_cnt": "ECE Flag Count",
    "down_up_ratio": "Down/Up Ratio",
    "pkt_size_avg": "Average Packet Size",
    "fwd_seg_size_avg": "Avg Fwd Segment Size",
    "bwd_seg_size_avg": "Avg Bwd Segment Size",
    "fwd_byts_b_avg": "Fwd Avg Bytes/Bulk",
    "fwd_pkts_b_avg": "Fwd Avg Packets/Bulk",
    "fwd_blk_rate_avg": "Fwd Avg Bulk Rate",
    "bwd_byts_b_avg": "Bwd Avg Bytes/Bulk",
    "bwd_pkts_b_avg": "Bwd Avg Packets/Bulk",
    "bwd_blk_rate_avg": "Bwd Avg Bulk Rate",
    "subflow_fwd_pkts": "Subflow Fwd Packets",
    "subflow_fwd_byts": "Subflow Fwd Bytes",
    "subflow_bwd_pkts": "Subflow Bwd Packets",
    "subflow_bwd_byts": "Subflow Bwd Bytes",
    "init_fwd_win_byts": "Init_Win_bytes_forward",
    "init_bwd_win_byts": "Init_Win_bytes_backward",
    "fwd_act_data_pkts": "act_data_pkt_fwd",
    "fwd_seg_size_min": "min_seg_size_forward",
    "active_mean": "Active Mean",
    "active_std": "Active Std",
    "active_max": "Active Max",
    "active_min": "Active Min",
    "idle_mean": "Idle Mean",
    "idle_std": "Idle Std",
    "idle_max": "Idle Max",
    "idle_min": "Idle Min",
}

In [ ]:

BENIGN_TOKENS = {
    "benign",
    "normal",
    "normal traffic",
    "non attack",
    "non-attack",
    "0",
}


def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = text.replace("\ufeff", "")
    text = text.replace("�", "-")
    text = text.strip().strip("'\"").casefold()
    text = text.rstrip(".")
    text = re.sub(r"[_/]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_label(value):
    text = normalize_text(value)

    if text in BENIGN_TOKENS or text.startswith("benign"):
        return "benign"

    if (
        "ddos" in text
        or "distributed denial" in text
        or "hoic" in text
        or "loic" in text
    ):
        return "ddos"

    if (
        text == "dos"
        or "denial of service" in text
        or "slowloris" in text
        or "slowhttptest" in text
        or "goldeneye" in text
        or "hulk" in text
    ):
        return "dos"

    if (
        "portscan" in text
        or "port scan" in text
        or "reconnaissance" in text
        or text == "recon"
        or text == "probe"
        or "nmap" in text
        or "ipsweep" in text
        or "portsweep" in text
        or "satan" in text
    ):
        return "reconnaissance"

    if (
        "brute force" in text
        or "bruteforce" in text
        or "patator" in text
        or "password" in text
    ):
        return "brute_force"

    if (
        "web attack" in text
        or "sql injection" in text
        or "xss" in text
        or "cross site" in text
    ):
        return "web_attack"

    if "bot" in text:
        return "botnet"

    if "infiltration" in text or "infilteration" in text:
        return "infiltration"

    if "heartbleed" in text:
        return "heartbleed"

    if "exploit" in text:
        return "exploits"

    if "fuzz" in text:
        return "fuzzers"

    if text == "generic" or "generic attack" in text:
        return "generic"

    if "backdoor" in text or "back door" in text:
        return "backdoor"

    if "shellcode" in text:
        return "shellcode"

    if "worm" in text:
        return "worms"

    if text == "r2l" or "remote to local" in text:
        return "r2l"

    if text == "u2r" or "user to root" in text:
        return "u2r"

    if text == "analysis":
        return "analysis"

    return "other_attack"


def normalize_exact_subtype(value):
    text = normalize_text(value)
    text = re.sub(
        r"(?:\s*[-]\s*|\s+)attempted\s*$",
        "",
        text,
    )
    return re.sub(r"\s+", " ", text).strip()

In [ ]:

def normalize_columns(frame):
    result = frame.copy()
    result.columns = (
        result.columns
        .astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.strip("'\"")
    )
    result = result.loc[
        :,
        ~result.columns.duplicated(keep="first")
    ].copy()
    return result


def find_label_column(frame, candidates):
    normalized_lookup = {
        normalize_text(column): column
        for column in frame.columns
    }

    for candidate in candidates:
        key = normalize_text(candidate)
        if key in normalized_lookup:
            return normalized_lookup[key]

    raise KeyError(
        f"No label column found. Candidates={candidates}; "
        f"available={frame.columns.tolist()}"
    )


def prepare_cic_schema_frame(
    raw_frame,
    source,
    label_candidates,
    source_file_column=None,
):
    frame = normalize_columns(raw_frame)

    frame = frame.rename(
        columns={
            **CICFLOWMETER_TO_CIC2017,
            **UNSW_TO_CIC2017,
        }
    )

    label_column = find_label_column(
        frame,
        label_candidates,
    )

    metadata = pd.DataFrame(index=frame.index)
    metadata[ORIGINAL_LABEL_COLUMN] = (
        frame[label_column]
        .astype("string")
        .str.strip()
    )
    metadata[SOURCE_COLUMN] = source

    if (
        source_file_column is not None
        and source_file_column in frame.columns
    ):
        metadata["source_file"] = (
            frame[source_file_column]
            .astype("string")
        )

    if (
        "Fwd Header Length.1"
        not in frame.columns
        and "Fwd Header Length" in frame.columns
    ):
        frame["Fwd Header Length.1"] = (
            frame["Fwd Header Length"]
        )

    missing = [
        column
        for column in CIC2017_FEATURE_COLUMNS
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            f"{source} is missing CIC-IDS2017 features: "
            f"{missing}"
        )

    features = frame[
        CIC2017_FEATURE_COLUMNS
    ].copy()

    for column in CIC2017_FEATURE_COLUMNS:
        features[column] = pd.to_numeric(
            features[column],
            errors="coerce",
        )

    features = features.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    result = pd.concat(
        [
            features.reset_index(drop=True),
            metadata.reset_index(drop=True),
        ],
        axis=1,
    )

    result[LABEL_COLUMN] = result[
        ORIGINAL_LABEL_COLUMN
    ].map(normalize_label)

    result[BINARY_COLUMN] = (
        result[LABEL_COLUMN].ne("benign")
    ).astype(np.int8)

    return result

In [ ]:

original_df = prepare_cic_schema_frame(
    original_raw,
    source="CIC-IDS2017-native",
    label_candidates=["Label", "label"],
    source_file_column="source_file",
)

darpa_pcap = prepare_cic_schema_frame(
    darpa_raw,
    source="DARPA",
    label_candidates=[
        "Attack_category",
        "attack_category",
        "Label",
        "label",
    ],
)

unsw_pcap = prepare_cic_schema_frame(
    unsw_raw,
    source="UNSW-NB15",
    label_candidates=[
        "category",
        "attack_cat",
        "Label",
        "label",
    ],
)

cic17_pcap = prepare_cic_schema_frame(
    cic17_raw,
    source="CIC-IDS2017",
    label_candidates=["Label", "label"],
)

cic18_pcap = prepare_cic_schema_frame(
    cic18_raw,
    source="CSE-CIC-IDS2018",
    label_candidates=["Label", "label"],
)

df_master = pd.concat(
    [
        darpa_pcap,
        unsw_pcap,
        cic17_pcap,
        cic18_pcap,
    ],
    ignore_index=True,
)

print("Native shape:", original_df.shape)
print("External master shape:", df_master.shape)
print("Feature count:", len(CIC2017_FEATURE_COLUMNS))

Native shape: (2830743, 83)
External master shape: (5647590, 82)
Feature count: 78


In [ ]:

schema_audit = pd.DataFrame(
    [
        {
            "dataset": name,
            "rows": len(frame),
            "features": len(CIC2017_FEATURE_COLUMNS),
            "missing_feature_cells": int(
                frame[CIC2017_FEATURE_COLUMNS]
                .isna()
                .sum()
                .sum()
            ),
            "benign_rows": int(
                (frame[BINARY_COLUMN] == 0).sum()
            ),
            "attack_rows": int(
                (frame[BINARY_COLUMN] == 1).sum()
            ),
        }
        for name, frame in {
            "CIC-IDS2017-native": original_df,
            "DARPA": darpa_pcap,
            "UNSW-NB15": unsw_pcap,
            "CIC-IDS2017": cic17_pcap,
            "CSE-CIC-IDS2018": cic18_pcap,
        }.items()
    ]
)

display(schema_audit)

label_audit = (
    pd.concat(
        [
            original_df,
            df_master,
        ],
        ignore_index=True,
    )
    .groupby(
        [SOURCE_COLUMN, LABEL_COLUMN],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)

display(label_audit)

,dataset,rows,features,missing_feature_cells,benign_rows,attack_rows
0,CIC-IDS2017-native,2830743,78,5734,2273097,557646
1,DARPA,2439149,78,0,2264038,175111
2,UNSW-NB15,313636,78,0,299097,14539
3,CIC-IDS2017,1233126,78,0,1055350,177776
4,CSE-CIC-IDS2018,1661679,78,30966,1540277,121402


,source,label,rows
0,CIC-IDS2017,benign,1055350
1,CIC-IDS2017,dos,177765
2,CIC-IDS2017,web_attack,11
3,CIC-IDS2017-native,benign,2273097
4,CIC-IDS2017-native,botnet,1966
5,CIC-IDS2017-native,brute_force,15342
6,CIC-IDS2017-native,ddos,128027
7,CIC-IDS2017-native,dos,252661
8,CIC-IDS2017-native,heartbleed,11
9,CIC-IDS2017-native,infiltration,36


In [ ]:

MODEL_ORIGINAL_TEST_COUNTS = {
    "kilichev_1dcnn": {
        "normal": 419_252,
        "attack": 84_908,
        "evidence": (
            "Confusion-matrix row totals: "
            "[415925 + 3327 normal, 3956 + 80952 attack]"
        ),
    },
    "buhl_dcnn": {
        "normal": 567_830,
        "attack": 139_139,
        "evidence": (
            "Confusion-matrix row totals: "
            "[566949 + 881 normal, 787 + 138352 attack]"
        ),
    },
    "kamal_transformer_dnn": {
        "normal": 14_035,
        "attack": 10_955,
        "evidence": (
            "Reported test class counts: "
            "14035 normal and 10955 attack"
        ),
    },
    "zkr_random_forest": {
        "normal": 454_265,
        "attack": 111_311,
        "evidence": (
            "Confusion-matrix row totals: "
            "[452407 + 1858 normal, 23899 + 87412 attack]"
        ),
    },
    "elmasry_dpso_dnn": {
        "normal": 18_750,
        "attack": 131_250,
        "evidence": (
            "Explicit test sampling: "
            "18750 normal and 131250 attack"
        ),
    },
}

MODEL_TEST_NORMAL_RATIO = {
    model_key: (
        counts["normal"]
        / (counts["normal"] + counts["attack"])
    )
    for model_key, counts
    in MODEL_ORIGINAL_TEST_COUNTS.items()
}

MODEL_TEST_ATTACK_RATIO = {
    model_key: 1.0 - normal_ratio
    for model_key, normal_ratio
    in MODEL_TEST_NORMAL_RATIO.items()
}

model_ratio_reference = pd.DataFrame(
    [
        {
            "model": model_key,
            "original_test_normal": counts["normal"],
            "original_test_attack": counts["attack"],
            "original_test_total": (
                counts["normal"] + counts["attack"]
            ),
            "target_normal_ratio": (
                MODEL_TEST_NORMAL_RATIO[model_key]
            ),
            "target_normal_pct": (
                100 * MODEL_TEST_NORMAL_RATIO[model_key]
            ),
            "target_attack_ratio": (
                MODEL_TEST_ATTACK_RATIO[model_key]
            ),
            "target_attack_pct": (
                100 * MODEL_TEST_ATTACK_RATIO[model_key]
            ),
            "evidence": counts["evidence"],
        }
        for model_key, counts
        in MODEL_ORIGINAL_TEST_COUNTS.items()
    ]
)

display(model_ratio_reference)


,model,original_test_normal,original_test_attack,original_test_total,target_normal_ratio,target_normal_pct,target_attack_ratio,target_attack_pct,evidence
0,kilichev_1dcnn,419252,84908,504160,0.831585,83.158521,0.168415,16.841479,Confusion-matrix row totals: [415925 + 3327 no...
1,buhl_dcnn,567830,139139,706969,0.803189,80.318939,0.196811,19.681061,Confusion-matrix row totals: [566949 + 881 nor...
2,kamal_transformer_dnn,14035,10955,24990,0.561625,56.162465,0.438375,43.837535,Reported test class counts: 14035 normal and 1...
3,zkr_random_forest,454265,111311,565576,0.803190,80.319002,0.196810,19.680998,Confusion-matrix row totals: [452407 + 1858 no...
4,elmasry_dpso_dnn,18750,131250,150000,0.125000,12.500000,0.875000,87.500000,Explicit test sampling: 18750 normal and 13125...


In [ ]:

NATIVE_TEST_FILES = {
    "Friday-Morning",
    "Friday-Afternoon-PortScan",
    "Friday-Afternoon-DDoS",
}


def split_source_frame(
    frame,
    test_size=0.20,
    seed=PARTITION_SEED,
):
    train_parts = []
    test_parts = []

    for binary_class, class_frame in frame.groupby(
        BINARY_COLUMN,
        sort=True,
    ):
        class_frame = class_frame.sample(
            frac=1,
            random_state=seed + int(binary_class),
        )

        if len(class_frame) == 1:
            train_parts.append(class_frame)
            continue

        n_test = max(
            1,
            int(round(len(class_frame) * test_size)),
        )
        n_test = min(n_test, len(class_frame) - 1)

        test_parts.append(class_frame.iloc[:n_test])
        train_parts.append(class_frame.iloc[n_test:])

    train_frame = (
        pd.concat(train_parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    test_frame = (
        pd.concat(test_parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return train_frame, test_frame


def balance_source_binary_training(frame, seed):
    working = frame.copy()

    counts = (
        working
        .groupby([SOURCE_COLUMN, BINARY_COLUMN])
        .size()
        .unstack(fill_value=0)
    )

    if not {0, 1}.issubset(counts.columns):
        raise ValueError(
            "Every included source must contain both classes."
        )

    quota = int(counts[[0, 1]].min().min())

    if quota < 1:
        raise ValueError(
            f"Invalid source/class quota: {quota}"
        )

    parts = []

    for source in sorted(
        working[SOURCE_COLUMN].unique()
    ):
        for binary_class in [0, 1]:
            cell = working[
                working[SOURCE_COLUMN].eq(source)
                & working[BINARY_COLUMN].eq(binary_class)
            ]

            parts.append(
                cell.sample(
                    n=quota,
                    replace=False,
                    random_state=seed,
                )
            )

    balanced = (
        pd.concat(parts, ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return balanced, quota


native_train = original_df[
    ~original_df["source_file"].isin(
        NATIVE_TEST_FILES
    )
].reset_index(drop=True)

native_test = original_df[
    original_df["source_file"].isin(
        NATIVE_TEST_FILES
    )
].reset_index(drop=True)

external_train_frames = {}
external_test_frames = {}

for source, source_frame in df_master.groupby(
    SOURCE_COLUMN,
    sort=True,
):
    train_part, test_part = split_source_frame(
        source_frame,
        test_size=0.20,
        seed=PARTITION_SEED,
    )
    external_train_frames[source] = train_part
    external_test_frames[source] = test_part

mixed_train_reference = pd.concat(
    [
        native_train,
        *external_train_frames.values(),
    ],
    ignore_index=True,
)

mixed_test_sets = {
    "CIC-IDS2017-native": native_test,
    **external_test_frames,
}

partition_audit = pd.DataFrame(
    [
        {
            "source": source,
            "train_rows": (
                len(native_train)
                if source == "CIC-IDS2017-native"
                else len(external_train_frames[source])
            ),
            "test_rows": len(test_frame),
            "test_benign": int(
                (test_frame[BINARY_COLUMN] == 0).sum()
            ),
            "test_attack": int(
                (test_frame[BINARY_COLUMN] == 1).sum()
            ),
        }
        for source, test_frame in mixed_test_sets.items()
    ]
)

display(partition_audit)

,source,train_rows,test_rows,test_benign,test_attack
0,CIC-IDS2017-native,2127498,703245,414322,288923
1,CIC-IDS2017,986501,246625,211070,35555
2,CSE-CIC-IDS2018,1329344,332335,308055,24280
3,DARPA,1951319,487830,452808,35022
4,UNSW-NB15,250909,62727,59819,2908


In [ ]:

def proportional_group_sample(
    frame,
    n_rows,
    group_column,
    seed,
):
    """Sample n_rows while retaining group proportions by largest remainder."""
    if n_rows < 0:
        raise ValueError("n_rows must be non-negative.")

    if n_rows == 0:
        return frame.iloc[0:0].copy()

    if n_rows >= len(frame):
        return (
            frame.sample(
                frac=1,
                random_state=seed,
            )
            .reset_index(drop=True)
        )

    working = frame.copy()
    group_values = (
        working[group_column]
        .astype("string")
        .fillna("<missing>")
    )

    counts = group_values.value_counts(
        dropna=False,
        sort=False,
    )

    exact = counts / counts.sum() * n_rows
    allocation = np.floor(exact).astype(int)
    allocation = allocation.clip(upper=counts)

    remaining = int(n_rows - allocation.sum())

    while remaining > 0:
        candidates = [
            group
            for group in counts.index
            if allocation[group] < counts[group]
        ]

        if not candidates:
            break

        candidates = sorted(
            candidates,
            key=lambda group: (
                exact[group] - allocation[group],
                counts[group] - allocation[group],
                str(group),
            ),
            reverse=True,
        )

        for group in candidates:
            if remaining == 0:
                break
            if allocation[group] < counts[group]:
                allocation[group] += 1
                remaining -= 1

    sampled_parts = []

    for offset, group in enumerate(counts.index):
        n_group = int(allocation[group])
        if n_group == 0:
            continue

        group_frame = working[
            group_values.eq(group)
        ]

        sampled_parts.append(
            group_frame.sample(
                n=n_group,
                replace=False,
                random_state=seed + offset + 1,
            )
        )

    sampled = pd.concat(
        sampled_parts,
        ignore_index=True,
    )

    return (
        sampled.sample(
            frac=1,
            random_state=seed,
        )
        .reset_index(drop=True)
    )


def match_model_test_ratio(
    frame,
    model_key,
    seed,
):
    """Downsample a source test set to the model's original binary ratio."""
    target_normal_ratio = (
        MODEL_TEST_NORMAL_RATIO[model_key]
    )
    target_attack_ratio = (
        MODEL_TEST_ATTACK_RATIO[model_key]
    )

    normal_frame = frame[
        frame[BINARY_COLUMN].eq(0)
    ].copy()

    attack_frame = frame[
        frame[BINARY_COLUMN].eq(1)
    ].copy()

    available_normal = len(normal_frame)
    available_attack = len(attack_frame)

    if available_normal < 1 or available_attack < 1:
        raise ValueError(
            "Ratio matching requires both normal and attack rows. "
            f"model={model_key}; normal={available_normal}; "
            f"attack={available_attack}"
        )

    maximum_total = int(
        math.floor(
            min(
                available_normal / target_normal_ratio,
                available_attack / target_attack_ratio,
            )
        )
    )

    if maximum_total < 2:
        raise ValueError(
            "Insufficient rows for a two-class matched test set."
        )

    while maximum_total >= 2:
        selected_normal = int(
            round(maximum_total * target_normal_ratio)
        )
        selected_attack = (
            maximum_total - selected_normal
        )

        if (
            selected_normal >= 1
            and selected_attack >= 1
            and selected_normal <= available_normal
            and selected_attack <= available_attack
        ):
            break

        maximum_total -= 1

    if maximum_total < 2:
        raise ValueError(
            "Could not construct a valid matched test set."
        )

    sampled_normal = normal_frame.sample(
        n=selected_normal,
        replace=False,
        random_state=seed,
    )

    sampled_attack = proportional_group_sample(
        attack_frame,
        n_rows=selected_attack,
        group_column=LABEL_COLUMN,
        seed=seed + 10_000,
    )

    matched = (
        pd.concat(
            [sampled_normal, sampled_attack],
            ignore_index=True,
        )
        .sample(
            frac=1,
            random_state=seed + 20_000,
        )
        .reset_index(drop=True)
    )

    actual_normal = int(
        matched[BINARY_COLUMN].eq(0).sum()
    )
    actual_attack = int(
        matched[BINARY_COLUMN].eq(1).sum()
    )

    audit = {
        "available_rows": int(len(frame)),
        "available_normal": int(available_normal),
        "available_attack": int(available_attack),
        "selected_rows": int(len(matched)),
        "selected_normal": actual_normal,
        "selected_attack": actual_attack,
        "target_normal_ratio": target_normal_ratio,
        "target_attack_ratio": target_attack_ratio,
        "target_normal_pct": 100 * target_normal_ratio,
        "target_attack_pct": 100 * target_attack_ratio,
        "actual_normal_ratio": (
            actual_normal / len(matched)
        ),
        "actual_attack_ratio": (
            actual_attack / len(matched)
        ),
        "actual_normal_pct": (
            100 * actual_normal / len(matched)
        ),
        "actual_attack_pct": (
            100 * actual_attack / len(matched)
        ),
    }

    return matched, audit


In [ ]:

def calculate_binary_metrics(
    y_true,
    y_pred,
    y_score=None,
):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )
    tn, fp, fn, tp = cm.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    metrics = {
        "n_test": int(len(y_true)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            y_pred,
        ),
        "false_positive_rate": (
            fp / (fp + tn)
            if (fp + tn) > 0
            else np.nan
        ),
        "roc_auc": np.nan,
    }

    if (
        y_score is not None
        and len(np.unique(y_true)) == 2
    ):
        metrics["roc_auc"] = roc_auc_score(
            y_true,
            np.asarray(y_score),
        )

    return metrics


def category_recall_table(
    frame,
    y_true,
    y_pred,
):
    categories = (
        frame[LABEL_COLUMN]
        .astype(str)
        .to_numpy()
    )

    rows = []

    for category in sorted(pd.unique(categories)):
        mask = categories == category
        category_true = y_true[mask]
        category_prediction = y_pred[mask]

        rows.append({
            "test_category": category,
            "support": int(mask.sum()),
            "attack_recall": (
                recall_score(
                    category_true,
                    category_prediction,
                    pos_label=1,
                    zero_division=0,
                )
                if np.any(category_true == 1)
                else np.nan
            ),
            "benign_specificity": (
                recall_score(
                    category_true,
                    category_prediction,
                    pos_label=0,
                    zero_division=0,
                )
                if np.any(category_true == 0)
                else np.nan
            ),
        })

    return pd.DataFrame(rows)


def subtype_recall_table(
    frame,
    y_true,
    y_pred,
):
    subtypes = (
        frame[ORIGINAL_LABEL_COLUMN]
        .astype(str)
        .str.strip()
        .str.rstrip(".")
        .to_numpy()
    )

    rows = []

    for subtype in sorted(pd.unique(subtypes)):
        mask = subtypes == subtype
        subtype_true = y_true[mask]
        subtype_prediction = y_pred[mask]

        rows.append({
            "test_subtype": subtype,
            "support": int(mask.sum()),
            "attack_recall": (
                recall_score(
                    subtype_true,
                    subtype_prediction,
                    pos_label=1,
                    zero_division=0,
                )
                if np.any(subtype_true == 1)
                else np.nan
            ),
            "benign_specificity": (
                recall_score(
                    subtype_true,
                    subtype_prediction,
                    pos_label=0,
                    zero_division=0,
                )
                if np.any(subtype_true == 0)
                else np.nan
            ),
        })

    return pd.DataFrame(rows)

In [ ]:

def numeric_matrix(frame, feature_columns):
    matrix = frame.reindex(
        columns=feature_columns,
    ).copy()

    for column in feature_columns:
        matrix[column] = pd.to_numeric(
            matrix[column],
            errors="coerce",
        )

    return matrix.replace(
        [np.inf, -np.inf],
        np.nan,
    )


def fit_numeric_preprocessor(
    train_frame,
    feature_columns,
    scaler_kind,
    drop_constant=True,
):
    matrix = numeric_matrix(
        train_frame,
        feature_columns,
    )

    medians = matrix.median(axis=0).fillna(0.0)
    matrix = matrix.fillna(medians)

    active_features = list(feature_columns)

    if drop_constant:
        active_features = [
            column
            for column in feature_columns
            if matrix[column].nunique(
                dropna=False
            ) > 1
        ]

    if not active_features:
        raise ValueError(
            "No non-constant training features remain."
        )

    matrix = matrix[active_features]

    if scaler_kind == "standard":
        scaler = StandardScaler()
    elif scaler_kind == "minmax":
        scaler = MinMaxScaler(
            feature_range=(0, 1)
        )
    elif scaler_kind == "none":
        scaler = None
    else:
        raise ValueError(
            f"Unknown scaler kind: {scaler_kind}"
        )

    transformed = (
        scaler.fit_transform(matrix)
        if scaler is not None
        else matrix.to_numpy()
    )

    return (
        {
            "feature_columns": active_features,
            "medians": medians,
            "scaler": scaler,
            "scaler_kind": scaler_kind,
        },
        np.asarray(transformed, dtype=np.float32),
    )


def transform_numeric_preprocessor(
    fitted,
    frame,
):
    matrix = numeric_matrix(
        frame,
        fitted["feature_columns"],
    )

    medians = fitted["medians"].reindex(
        fitted["feature_columns"]
    ).fillna(0.0)

    matrix = matrix.fillna(medians)

    transformed = (
        fitted["scaler"].transform(matrix)
        if fitted["scaler"] is not None
        else matrix.to_numpy()
    )

    return np.asarray(
        transformed,
        dtype=np.float32,
    )


def prepare_test(frame, seed):
    return optional_sample(
        frame,
        MAX_TEST_ROWS,
        seed,
    )

In [ ]:

KILICHEV_DROPPED_FEATURES = {
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate",
}


def build_kilichev_1dcnn(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(n_features, 1)
        ),
        tf.keras.layers.Conv1D(
            filters=64,
            kernel_size=9,
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=5,
        ),
        tf.keras.layers.Conv1D(
            filters=128,
            kernel_size=9,
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=2,
        ),
        tf.keras.layers.Dropout(0.113585),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            256,
            activation="relu",
        ),
        tf.keras.layers.Dropout(0.113585),
        tf.keras.layers.Dense(
            256,
            activation="relu",
        ),
        tf.keras.layers.Dropout(0.113585),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001193
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    return model


def fit_kilichev_1dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    feature_columns = [
        column
        for column in CIC2017_FEATURE_COLUMNS
        if column not in KILICHEV_DROPPED_FEATURES
    ]

    train_part, validation_part = train_test_split(
        train_frame,
        test_size=0.125,
        random_state=seed,
        shuffle=True,
        stratify=train_frame[BINARY_COLUMN],
    )

    preprocessor, X_train = (
        fit_numeric_preprocessor(
            train_part,
            feature_columns,
            scaler_kind="standard",
            drop_constant=False,
        )
    )

    X_validation = (
        transform_numeric_preprocessor(
            preprocessor,
            validation_part,
        )
    )

    model = build_kilichev_1dcnn(
        X_train.shape[1]
    )

    model.fit(
        X_train[..., np.newaxis],
        binary_y(train_part),
        validation_data=(
            X_validation[..., np.newaxis],
            binary_y(validation_part),
        ),
        epochs=EPOCHS["kilichev_1dcnn"],
        batch_size=BATCH_SIZES[
            "kilichev_1dcnn"
        ],
        shuffle=True,
        callbacks=make_callbacks(),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_kilichev_1dcnn(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[..., np.newaxis],
            batch_size=BATCH_SIZES[
                "kilichev_1dcnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= 0.5
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def build_buhl_dcnn(n_features):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(n_features, 1)
        ),
        tf.keras.layers.Conv1D(
            256,
            3,
            strides=1,
            padding="valid",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(
            pool_size=3,
            strides=1,
            padding="valid",
        ),
        tf.keras.layers.Conv1D(
            128,
            3,
            strides=1,
            padding="valid",
            activation="relu",
        ),
        tf.keras.layers.UpSampling1D(
            size=3,
        ),
        tf.keras.layers.Conv1D(
            256,
            3,
            strides=1,
            padding="valid",
            activation="relu",
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            256,
            activation="relu",
        ),
        tf.keras.layers.Dropout(0.1),
        tf.keras.layers.Dense(
            128,
            activation="relu",
        ),
        tf.keras.layers.Dropout(0.1),
        tf.keras.layers.Dense(
            128,
            activation="relu",
        ),
        tf.keras.layers.Dropout(0.1),
        tf.keras.layers.Dense(
            1,
            activation="sigmoid",
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.0001
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


def fit_buhl_dcnn(train_frame, seed):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_numeric_preprocessor(
            train_frame,
            CIC2017_FEATURE_COLUMNS,
            scaler_kind="standard",
            drop_constant=True,
        )
    )

    model = build_buhl_dcnn(
        X_train.shape[1]
    )

    model.fit(
        X_train[..., np.newaxis],
        binary_y(train_frame),
        validation_split=0.10,
        epochs=EPOCHS["buhl_dcnn"],
        batch_size=BATCH_SIZES["buhl_dcnn"],
        shuffle=True,
        callbacks=make_callbacks(),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_buhl_dcnn(fitted, test_frame):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[..., np.newaxis],
            batch_size=BATCH_SIZES[
                "buhl_dcnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= 0.5
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def build_kamal_transformer_dnn(n_features):
    model_input = tf.keras.Input(
        shape=(n_features, 1),
        name="transformer_input",
    )

    attention_output = (
        tf.keras.layers.MultiHeadAttention(
            num_heads=8,
            key_dim=64,
            output_shape=1,
            name="multi_head_attention",
        )(
            model_input,
            model_input,
        )
    )

    attention_output = (
        tf.keras.layers.LayerNormalization(
            epsilon=1e-6,
        )(attention_output)
    )

    attention_residual = tf.keras.layers.Add()([
        model_input,
        attention_output,
    ])

    feed_forward = tf.keras.layers.Dense(
        512,
        activation="relu",
    )(attention_residual)

    feed_forward = tf.keras.layers.Dense(
        512,
        activation=None,
    )(feed_forward)

    projected_residual = tf.keras.layers.Dense(
        512,
        activation=None,
    )(attention_residual)

    transformer_output = tf.keras.layers.Add()([
        projected_residual,
        feed_forward,
    ])

    transformer_output = (
        tf.keras.layers.LayerNormalization(
            epsilon=1e-6,
        )(transformer_output)
    )

    x = tf.keras.layers.Flatten()(
        transformer_output
    )
    x = tf.keras.layers.Dense(
        256,
        activation="relu",
    )(x)
    x = tf.keras.layers.Dropout(1e-7)(x)
    x = tf.keras.layers.Dense(
        128,
        activation="relu",
    )(x)
    x = tf.keras.layers.Dropout(1e-7)(x)
    x = tf.keras.layers.Dense(
        64,
        activation="relu",
    )(x)

    output = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
    )(x)

    model = tf.keras.Model(
        model_input,
        output,
        name="kamal_transformer_dnn",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=["accuracy"],
    )

    return model


def kamal_clean_and_resample(
    X_train,
    y_train,
    seed,
):
    X_result = np.asarray(
        X_train,
        dtype=np.float32,
    )
    y_result = np.asarray(
        y_train,
        dtype=np.int8,
    )

    if (
        KAMAL_USE_LOF
        and len(X_result) > 20
    ):
        lof = LocalOutlierFactor(
            n_neighbors=20,
            contamination=0.05,
            novelty=False,
            n_jobs=-1,
        )
        inlier_mask = (
            lof.fit_predict(X_result) == 1
        )

        if len(np.unique(y_result[inlier_mask])) == 2:
            X_result = X_result[inlier_mask]
            y_result = y_result[inlier_mask]

    if not KAMAL_USE_ADASYN_SMOTE_ENN:
        return X_result, y_result

    counts = pd.Series(y_result).value_counts()

    if (
        len(counts) == 2
        and counts.min() >= 6
        and counts.min() < counts.max()
    ):
        try:
            X_result, y_result = ADASYN(
                sampling_strategy="auto",
                random_state=seed,
                n_neighbors=5,
            ).fit_resample(
                X_result,
                y_result,
            )
        except ValueError as error:
            warnings.warn(
                f"ADASYN skipped: {error}"
            )

    counts = pd.Series(y_result).value_counts()

    if (
        len(counts) == 2
        and counts.min() >= 6
        and counts.min() < counts.max()
    ):
        minority_label = int(counts.idxmin())
        majority_count = int(counts.max())

        try:
            X_result, y_result = SMOTE(
                sampling_strategy={
                    minority_label: majority_count
                },
                random_state=seed,
                k_neighbors=5,
            ).fit_resample(
                X_result,
                y_result,
            )
        except ValueError as error:
            warnings.warn(
                f"SMOTE skipped: {error}"
            )

    if len(X_result) > 3:
        try:
            X_result, y_result = (
                EditedNearestNeighbours(
                    sampling_strategy="all",
                    n_neighbors=3,
                    kind_sel="all",
                    n_jobs=-1,
                )
                .fit_resample(
                    X_result,
                    y_result,
                )
            )
        except ValueError as error:
            warnings.warn(
                f"ENN skipped: {error}"
            )

    return (
        np.asarray(X_result, dtype=np.float32),
        np.asarray(y_result, dtype=np.int8),
    )


def fit_kamal_transformer_dnn(
    train_frame,
    seed,
):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        KAMAL_MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_numeric_preprocessor(
            train_frame,
            CIC2017_FEATURE_COLUMNS,
            scaler_kind="minmax",
            drop_constant=True,
        )
    )

    X_train, y_train = (
        kamal_clean_and_resample(
            X_train,
            binary_y(train_frame),
            seed,
        )
    )

    model = build_kamal_transformer_dnn(
        X_train.shape[1]
    )

    model.fit(
        X_train[..., np.newaxis],
        y_train,
        validation_split=0.10,
        epochs=EPOCHS[
            "kamal_transformer_dnn"
        ],
        batch_size=BATCH_SIZES[
            "kamal_transformer_dnn"
        ],
        shuffle=True,
        callbacks=make_callbacks(),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_kamal_transformer_dnn(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probability = (
        fitted["model"]
        .predict(
            X_test[..., np.newaxis],
            batch_size=BATCH_SIZES[
                "kamal_transformer_dnn"
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    prediction = (
        probability >= 0.5
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

def fit_zkr_random_forest(
    train_frame,
    seed,
):
    set_all_seeds(seed)

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train_z = (
        fit_numeric_preprocessor(
            train_frame,
            CIC2017_FEATURE_COLUMNS,
            scaler_kind="standard",
            drop_constant=True,
        )
    )

    y_train = binary_y(train_frame)

    inlier_mask = (
        np.abs(X_train_z) <= 3
    ).all(axis=1)

    if (
        inlier_mask.sum() >= 20
        and len(np.unique(y_train[inlier_mask])) == 2
    ):
        X_inliers = X_train_z[inlier_mask]
        y_inliers = y_train[inlier_mask]
    else:
        X_inliers = X_train_z
        y_inliers = y_train

    n_components = min(
        10,
        X_inliers.shape[1],
        X_inliers.shape[0] - 1,
    )

    pca = PCA(
        n_components=n_components,
        random_state=seed,
    )

    X_pca = pca.fit_transform(
        X_inliers
    ).astype(np.float32)

    kmeans = KMeans(
        n_clusters=2,
        random_state=seed,
        n_init=10,
    )

    cluster_ids = kmeans.fit_predict(
        X_pca
    )

    X_zkr = np.column_stack([
        X_pca,
        cluster_ids,
    ]).astype(np.float32)

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_zkr,
        y_inliers,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "pca": pca,
        "kmeans": kmeans,
        "seed": seed,
    }


def predict_zkr_random_forest(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test_z = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    X_test_pca = fitted["pca"].transform(
        X_test_z
    ).astype(np.float32)

    cluster_ids = fitted["kmeans"].predict(
        X_test_pca
    )

    X_test_zkr = np.column_stack([
        X_test_pca,
        cluster_ids,
    ]).astype(np.float32)

    prediction = fitted["model"].predict(
        X_test_zkr
    ).astype(np.int8)

    probability = fitted["model"].predict_proba(
        X_test_zkr
    )[:, 1]

    return (
        binary_y(evaluated),
        prediction,
        probability,
        evaluated,
    )

In [ ]:

ELMASRY_ONE_BASED_FEATURE_INDICES = [
    8,
    11,
    15,
    18,
    20,
    23,
    24,
    26,
    28,
    31,
    33,
    37,
    43,
    44,
    51,
    53,
    54,
    59,
    70,
    73,
    74,
    77,
]

ELMASRY_FEATURE_COLUMNS = [
    CIC2017_FEATURE_COLUMNS[index - 1]
    for index in ELMASRY_ONE_BASED_FEATURE_INDICES
]


def build_elmasry_dpso_dnn():
    learning_rate_schedule = (
        tf.keras.optimizers.schedules.InverseTimeDecay(
            initial_learning_rate=0.1,
            decay_steps=1,
            decay_rate=0.005,
            staircase=False,
        )
    )

    model = tf.keras.Sequential([
        tf.keras.layers.Input(
            shape=(22,)
        ),
        tf.keras.layers.Dense(
            23,
            activation="tanh",
            kernel_initializer=(
                tf.keras.initializers.RandomUniform(
                    minval=-0.05,
                    maxval=0.05,
                    seed=0,
                )
            ),
        ),
        tf.keras.layers.Dropout(
            0.25,
            seed=0,
        ),
        tf.keras.layers.Dense(
            15,
            activation="softsign",
            kernel_initializer=(
                tf.keras.initializers.RandomUniform(
                    minval=-0.05,
                    maxval=0.05,
                    seed=1,
                )
            ),
        ),
        tf.keras.layers.Dropout(
            0.25,
            seed=1,
        ),
        tf.keras.layers.Dense(
            6,
            activation="softplus",
            kernel_initializer=(
                tf.keras.initializers.RandomUniform(
                    minval=-0.05,
                    maxval=0.05,
                    seed=2,
                )
            ),
        ),
        tf.keras.layers.Dropout(
            0.25,
            seed=2,
        ),
        tf.keras.layers.Dense(
            2,
            activation="softmax",
            kernel_initializer=(
                tf.keras.initializers.RandomUniform(
                    minval=-0.05,
                    maxval=0.05,
                    seed=3,
                )
            ),
        ),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate_schedule,
            momentum=0.3,
        ),
        loss=(
            tf.keras.losses
            .SparseCategoricalCrossentropy()
        ),
        metrics=["accuracy"],
    )

    return model


def fit_elmasry_dpso_dnn(
    train_frame,
    seed,
):
    set_all_seeds(seed)
    tf.keras.backend.clear_session()

    train_frame = optional_sample(
        train_frame,
        MAX_TRAIN_ROWS,
        seed,
    )

    preprocessor, X_train = (
        fit_numeric_preprocessor(
            train_frame,
            ELMASRY_FEATURE_COLUMNS,
            scaler_kind="minmax",
            drop_constant=False,
        )
    )

    model = build_elmasry_dpso_dnn()

    model.fit(
        X_train,
        binary_y(train_frame),
        validation_split=0.10,
        epochs=EPOCHS[
            "elmasry_dpso_dnn"
        ],
        batch_size=BATCH_SIZES[
            "elmasry_dpso_dnn"
        ],
        shuffle=True,
        callbacks=make_callbacks(include_reduce_lr=False),
        verbose=2,
    )

    return {
        "model": model,
        "preprocessor": preprocessor,
        "seed": seed,
    }


def predict_elmasry_dpso_dnn(
    fitted,
    test_frame,
):
    evaluated = prepare_test(
        test_frame,
        fitted["seed"],
    )

    X_test = transform_numeric_preprocessor(
        fitted["preprocessor"],
        evaluated,
    )

    probabilities = fitted["model"].predict(
        X_test,
        batch_size=BATCH_SIZES[
            "elmasry_dpso_dnn"
        ],
        verbose=0,
    )

    prediction = np.argmax(
        probabilities,
        axis=1,
    ).astype(np.int8)

    return (
        binary_y(evaluated),
        prediction,
        probabilities[:, 1],
        evaluated,
    )

In [ ]:

MODEL_REGISTRY = {
    "kilichev_1dcnn": {
        "fit": fit_kilichev_1dcnn,
        "predict": predict_kilichev_1dcnn,
    },
    "buhl_dcnn": {
        "fit": fit_buhl_dcnn,
        "predict": predict_buhl_dcnn,
    },
    "kamal_transformer_dnn": {
        "fit": fit_kamal_transformer_dnn,
        "predict": predict_kamal_transformer_dnn,
    },
    "zkr_random_forest": {
        "fit": fit_zkr_random_forest,
        "predict": predict_zkr_random_forest,
    },
    "elmasry_dpso_dnn": {
        "fit": fit_elmasry_dpso_dnn,
        "predict": predict_elmasry_dpso_dnn,
    },
}

missing_models = [
    model
    for model in MODEL_KEYS
    if model not in MODEL_REGISTRY
]

if missing_models:
    raise KeyError(
        f"Unregistered models: {missing_models}"
    )

print("Registered models:", list(MODEL_REGISTRY))

Registered models: ['kilichev_1dcnn', 'buhl_dcnn', 'kamal_transformer_dnn', 'zkr_random_forest', 'elmasry_dpso_dnn']


In [ ]:

def merge_and_save(
    path,
    new_rows,
    key_columns,
):
    path = Path(path)

    if path.exists():
        existing = pd.read_csv(path)
        combined = pd.concat(
            [existing, new_rows],
            ignore_index=True,
        )
    else:
        combined = new_rows.copy()

    combined = (
        combined
        .drop_duplicates(
            subset=key_columns,
            keep="last",
        )
        .reset_index(drop=True)
    )

    combined.to_csv(
        path,
        index=False,
    )

    return combined

In [ ]:

MIXED_EXPERIMENT = (
    "exp4_mixed_traffic_model_specific_test_ratios"
)
TRAINING_VARIANT = (
    "balanced_mixed_train_model_ratio_test"
)


def evaluate_mixed_source_model(
    model_key,
    seed,
):
    adapter = MODEL_REGISTRY[model_key]

    train_frame, source_class_quota = (
        balance_source_binary_training(
            mixed_train_reference,
            seed,
        )
    )

    target_normal_ratio = (
        MODEL_TEST_NORMAL_RATIO[model_key]
    )
    target_attack_ratio = (
        MODEL_TEST_ATTACK_RATIO[model_key]
    )

    print(
        f"\n{MIXED_EXPERIMENT} | "
        f"{model_key} | seed={seed}",
        flush=True,
    )
    print(
        f"  Target test ratio: "
        f"{100 * target_normal_ratio:.4f}% normal / "
        f"{100 * target_attack_ratio:.4f}% attack",
        flush=True,
    )
    print(
        f"  Balanced training rows: "
        f"{len(train_frame):,}; "
        f"quota={source_class_quota:,}",
        flush=True,
    )

    fit_started = time.perf_counter()
    fitted = adapter["fit"](
        train_frame,
        seed,
    )
    fit_seconds = (
        time.perf_counter() - fit_started
    )

    overall_rows = []
    category_frames = []
    subtype_frames = []
    ratio_rows = []

    pooled_true = []
    pooled_prediction = []
    pooled_score = []

    source_items = list(
        mixed_test_sets.items()
    )

    for source_index, (
        test_source,
        raw_test_frame,
    ) in enumerate(source_items):
        ratio_seed = (
            seed * 100_000
            + MODEL_KEYS.index(model_key) * 1_000
            + source_index
        )

        matched_test_frame, ratio_audit = (
            match_model_test_ratio(
                raw_test_frame,
                model_key=model_key,
                seed=ratio_seed,
            )
        )

        print(
            f"  Testing {test_source}: "
            f"{len(raw_test_frame):,} available -> "
            f"{len(matched_test_frame):,} matched; "
            f"normal={ratio_audit['actual_normal_pct']:.4f}%; "
            f"attack={ratio_audit['actual_attack_pct']:.4f}%",
            flush=True,
        )

        prediction_started = time.perf_counter()

        (
            y_true,
            y_pred,
            y_score,
            evaluated,
        ) = adapter["predict"](
            fitted,
            matched_test_frame,
        )

        prediction_seconds = (
            time.perf_counter()
            - prediction_started
        )

        metrics = calculate_binary_metrics(
            y_true,
            y_pred,
            y_score,
        )

        evaluated_normal = int(
            np.sum(y_true == 0)
        )
        evaluated_attack = int(
            np.sum(y_true == 1)
        )

        overall_rows.append({
            "experiment": MIXED_EXPERIMENT,
            "training_variant": TRAINING_VARIANT,
            "model": model_key,
            "seed": seed,
            "test_source": test_source,
            "source_class_quota": (
                source_class_quota
            ),
            "n_train_used": len(train_frame),
            "n_test_raw": len(raw_test_frame),
            "n_test_ratio_matched": (
                len(matched_test_frame)
            ),
            "n_test_evaluated": len(y_true),
            "target_normal_ratio": (
                target_normal_ratio
            ),
            "target_attack_ratio": (
                target_attack_ratio
            ),
            "actual_normal_ratio": (
                evaluated_normal / len(y_true)
            ),
            "actual_attack_ratio": (
                evaluated_attack / len(y_true)
            ),
            "test_normal_pct": (
                100 * evaluated_normal / len(y_true)
            ),
            "test_attack_pct": (
                100 * evaluated_attack / len(y_true)
            ),
            "fit_seconds": fit_seconds,
            "prediction_seconds": (
                prediction_seconds
            ),
            **metrics,
        })

        ratio_rows.append({
            "experiment": MIXED_EXPERIMENT,
            "training_variant": TRAINING_VARIANT,
            "model": model_key,
            "seed": seed,
            "test_source": test_source,
            **ratio_audit,
        })

        category_table = category_recall_table(
            evaluated,
            y_true,
            y_pred,
        )
        category_table["experiment"] = (
            MIXED_EXPERIMENT
        )
        category_table["training_variant"] = (
            TRAINING_VARIANT
        )
        category_table["model"] = model_key
        category_table["seed"] = seed
        category_table["test_source"] = (
            test_source
        )
        category_frames.append(category_table)

        subtype_table = subtype_recall_table(
            evaluated,
            y_true,
            y_pred,
        )
        subtype_table["experiment"] = (
            MIXED_EXPERIMENT
        )
        subtype_table["training_variant"] = (
            TRAINING_VARIANT
        )
        subtype_table["model"] = model_key
        subtype_table["seed"] = seed
        subtype_table["test_source"] = (
            test_source
        )
        subtype_frames.append(subtype_table)

        pooled_true.append(y_true)
        pooled_prediction.append(y_pred)
        pooled_score.append(y_score)

    pooled_true_array = np.concatenate(
        pooled_true
    )
    pooled_prediction_array = np.concatenate(
        pooled_prediction
    )
    pooled_score_array = np.concatenate(
        pooled_score
    )

    pooled_metrics = calculate_binary_metrics(
        pooled_true_array,
        pooled_prediction_array,
        pooled_score_array,
    )

    pooled_normal = int(
        np.sum(pooled_true_array == 0)
    )
    pooled_attack = int(
        np.sum(pooled_true_array == 1)
    )

    overall_rows.append({
        "experiment": MIXED_EXPERIMENT,
        "training_variant": TRAINING_VARIANT,
        "model": model_key,
        "seed": seed,
        "test_source": "POOLED_ALL_ROWS",
        "source_class_quota": (
            source_class_quota
        ),
        "n_train_used": len(train_frame),
        "n_test_raw": int(
            sum(
                len(frame)
                for frame in mixed_test_sets.values()
            )
        ),
        "n_test_ratio_matched": int(
            sum(len(values) for values in pooled_true)
        ),
        "n_test_evaluated": int(
            len(pooled_true_array)
        ),
        "target_normal_ratio": (
            target_normal_ratio
        ),
        "target_attack_ratio": (
            target_attack_ratio
        ),
        "actual_normal_ratio": (
            pooled_normal / len(pooled_true_array)
        ),
        "actual_attack_ratio": (
            pooled_attack / len(pooled_true_array)
        ),
        "test_normal_pct": (
            100 * pooled_normal / len(pooled_true_array)
        ),
        "test_attack_pct": (
            100 * pooled_attack / len(pooled_true_array)
        ),
        "fit_seconds": fit_seconds,
        "prediction_seconds": np.nan,
        **pooled_metrics,
    })

    del fitted
    gc.collect()
    tf.keras.backend.clear_session()

    return (
        pd.DataFrame(overall_rows),
        pd.concat(
            category_frames,
            ignore_index=True,
        ),
        pd.concat(
            subtype_frames,
            ignore_index=True,
        ),
        pd.DataFrame(ratio_rows),
    )


In [ ]:

def run_mixed_source_experiment():
    overall_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_all_runs_partial.csv"
    )
    category_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_category_runs_partial.csv"
    )
    subtype_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_subtype_runs_partial.csv"
    )
    ratio_path = (
        RESULTS_DIR
        / f"{MIXED_EXPERIMENT}_test_ratio_manifest_partial.csv"
    )

    completed = (
        pd.read_csv(overall_path)
        if overall_path.exists()
        else pd.DataFrame()
    )

    overall_accumulated = completed.copy()
    category_accumulated = (
        pd.read_csv(category_path)
        if category_path.exists()
        else pd.DataFrame()
    )
    subtype_accumulated = (
        pd.read_csv(subtype_path)
        if subtype_path.exists()
        else pd.DataFrame()
    )
    ratio_accumulated = (
        pd.read_csv(ratio_path)
        if ratio_path.exists()
        else pd.DataFrame()
    )

    for model_key in MODEL_KEYS:
        for seed in SEEDS:
            if not completed.empty:
                complete_mask = (
                    completed["experiment"].eq(
                        MIXED_EXPERIMENT
                    )
                    & completed["model"].eq(model_key)
                    & completed["seed"].eq(seed)
                    & completed["test_source"].eq(
                        "POOLED_ALL_ROWS"
                    )
                )
                is_complete = bool(
                    complete_mask.any()
                )
            else:
                is_complete = False

            if is_complete:
                print(
                    f"SKIP completed: "
                    f"{model_key}, seed={seed}",
                    flush=True,
                )
                continue

            (
                overall,
                categories,
                subtypes,
                ratio_manifest,
            ) = evaluate_mixed_source_model(
                model_key,
                seed,
            )

            category_accumulated = merge_and_save(
                category_path,
                categories,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                    "test_category",
                ],
            )

            subtype_accumulated = merge_and_save(
                subtype_path,
                subtypes,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                    "test_subtype",
                ],
            )

            ratio_accumulated = merge_and_save(
                ratio_path,
                ratio_manifest,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                ],
            )

            overall_accumulated = merge_and_save(
                overall_path,
                overall,
                [
                    "experiment",
                    "training_variant",
                    "model",
                    "seed",
                    "test_source",
                ],
            )

            completed = overall_accumulated

    return (
        overall_accumulated,
        category_accumulated,
        subtype_accumulated,
        ratio_accumulated,
    )


In [25]:
(
    mixed_overall,
    mixed_categories,
    mixed_subtypes,
    mixed_ratio_manifest,
) = run_mixed_source_experiment()


SKIP completed: kilichev_1dcnn, seed=11
SKIP completed: kilichev_1dcnn, seed=22
SKIP completed: kilichev_1dcnn, seed=33
SKIP completed: kilichev_1dcnn, seed=44
SKIP completed: kilichev_1dcnn, seed=55
SKIP completed: buhl_dcnn, seed=11
SKIP completed: buhl_dcnn, seed=22
SKIP completed: buhl_dcnn, seed=33
SKIP completed: buhl_dcnn, seed=44
SKIP completed: buhl_dcnn, seed=55
SKIP completed: kamal_transformer_dnn, seed=11
SKIP completed: kamal_transformer_dnn, seed=22
SKIP completed: kamal_transformer_dnn, seed=33
SKIP completed: kamal_transformer_dnn, seed=44
SKIP completed: kamal_transformer_dnn, seed=55
SKIP completed: zkr_random_forest, seed=11
SKIP completed: zkr_random_forest, seed=22
SKIP completed: zkr_random_forest, seed=33
SKIP completed: zkr_random_forest, seed=44
SKIP completed: zkr_random_forest, seed=55

exp4_mixed_traffic_model_specific_test_ratios | elmasry_dpso_dnn | seed=11
  Target test ratio: 12.5000% normal / 87.5000% attack
  Balanced training rows: 116,310; quota=11,

In [ ]:

MIXED_METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "macro_f1",
    "mcc",
    "false_positive_rate",
    "roc_auc",
]

mixed_per_source_summary = (
    mixed_overall
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "test_source",
        ],
        dropna=False,
    )[MIXED_METRICS]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

mixed_source_macro_by_seed = (
    mixed_overall[
        mixed_overall["test_source"].ne(
            "POOLED_ALL_ROWS"
        )
    ]
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "seed",
        ],
        dropna=False,
    )
    .agg(
        source_macro_f1=("f1", "mean"),
        source_macro_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        source_macro_recall=("recall", "mean"),
        source_macro_specificity=(
            "specificity",
            "mean",
        ),
    )
    .reset_index()
)

mixed_source_macro_summary = (
    mixed_source_macro_by_seed
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
        ],
        dropna=False,
    )[
        [
            "source_macro_f1",
            "source_macro_balanced_accuracy",
            "source_macro_recall",
            "source_macro_specificity",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
)

mixed_category_summary = (
    mixed_categories
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "test_source",
            "test_category",
        ],
        dropna=False,
    )
    .agg(
        support=("support", "first"),
        attack_recall_mean=(
            "attack_recall",
            "mean",
        ),
        attack_recall_std=(
            "attack_recall",
            "std",
        ),
        benign_specificity_mean=(
            "benign_specificity",
            "mean",
        ),
        benign_specificity_std=(
            "benign_specificity",
            "std",
        ),
    )
    .reset_index()
)

ratio_summary = (
    mixed_ratio_manifest
    .groupby(
        [
            "experiment",
            "training_variant",
            "model",
            "test_source",
        ],
        dropna=False,
    )
    .agg(
        target_normal_pct=(
            "target_normal_pct",
            "first",
        ),
        target_attack_pct=(
            "target_attack_pct",
            "first",
        ),
        available_rows=(
            "available_rows",
            "first",
        ),
        selected_rows_mean=(
            "selected_rows",
            "mean",
        ),
        actual_normal_pct_mean=(
            "actual_normal_pct",
            "mean",
        ),
        actual_normal_pct_min=(
            "actual_normal_pct",
            "min",
        ),
        actual_normal_pct_max=(
            "actual_normal_pct",
            "max",
        ),
        actual_attack_pct_mean=(
            "actual_attack_pct",
            "mean",
        ),
    )
    .reset_index()
)

model_ratio_reference.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_model_ratio_reference.csv",
    index=False,
)
mixed_overall.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_all_runs.csv",
    index=False,
)
mixed_categories.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_category_runs.csv",
    index=False,
)
mixed_subtypes.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_subtype_runs.csv",
    index=False,
)
mixed_ratio_manifest.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_test_ratio_manifest.csv",
    index=False,
)
mixed_per_source_summary.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_per_source_summary.csv",
    index=False,
)
mixed_source_macro_by_seed.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_source_macro_by_seed.csv",
    index=False,
)
mixed_source_macro_summary.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_source_macro_summary.csv",
    index=False,
)
mixed_category_summary.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_category_summary.csv",
    index=False,
)
ratio_summary.to_csv(
    RESULTS_DIR
    / f"{MIXED_EXPERIMENT}_test_ratio_summary.csv",
    index=False,
)

display(model_ratio_reference)
display(ratio_summary)
display(mixed_per_source_summary)
display(mixed_source_macro_summary)
display(mixed_category_summary)


,model,original_test_normal,original_test_attack,original_test_total,target_normal_ratio,target_normal_pct,target_attack_ratio,target_attack_pct,evidence
0,kilichev_1dcnn,419252,84908,504160,0.831585,83.158521,0.168415,16.841479,Confusion-matrix row totals: [415925 + 3327 no...
1,buhl_dcnn,567830,139139,706969,0.803189,80.318939,0.196811,19.681061,Confusion-matrix row totals: [566949 + 881 nor...
2,kamal_transformer_dnn,14035,10955,24990,0.561625,56.162465,0.438375,43.837535,Reported test class counts: 14035 normal and 1...
3,zkr_random_forest,454265,111311,565576,0.803190,80.319002,0.196810,19.680998,Confusion-matrix row totals: [452407 + 1858 no...
4,elmasry_dpso_dnn,18750,131250,150000,0.125000,12.500000,0.875000,87.500000,Explicit test sampling: 18750 normal and 13125...


,experiment,training_variant,model,test_source,target_normal_pct,target_attack_pct,available_rows,selected_rows_mean,actual_normal_pct_mean,actual_normal_pct_min,actual_normal_pct_max,actual_attack_pct_mean
0,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017,80.318939,19.681061,246625,180655.0,80.318840,80.318840,80.318840,19.681160
1,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017-native,80.318939,19.681061,703245,515845.0,80.318894,80.318894,80.318894,19.681106
2,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CSE-CIC-IDS2018,80.318939,19.681061,332335,123367.0,80.318886,80.318886,80.318886,19.681114
3,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,DARPA,80.318939,19.681061,487830,177947.0,80.318859,80.318859,80.318859,19.681141
4,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,UNSW-NB15,80.318939,19.681061,62727,14775.0,80.318105,80.318105,80.318105,19.681895
5,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,elmasry_dpso_dnn,CIC-IDS2017,12.500000,87.500000,246625,40634.0,12.499385,12.499385,12.499385,87.500615
6,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,elmasry_dpso_dnn,CIC-IDS2017-native,12.500000,87.500000,703245,330197.0,12.500114,12.500114,12.500114,87.499886
7,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,elmasry_dpso_dnn,CSE-CIC-IDS2018,12.500000,87.500000,332335,27748.0,12.498198,12.498198,12.498198,87.501802
8,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,elmasry_dpso_dnn,DARPA,12.500000,87.500000,487830,40025.0,12.499688,12.499688,12.499688,87.500312
9,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,elmasry_dpso_dnn,UNSW-NB15,12.500000,87.500000,62727,3323.0,12.488715,12.488715,12.488715,87.511285


experiment  \
                                                    
0   exp4_mixed_traffic_model_specific_test_ratios   
1   exp4_mixed_traffic_model_specific_test_ratios   
2   exp4_mixed_traffic_model_specific_test_ratios   
3   exp4_mixed_traffic_model_specific_test_ratios   
4   exp4_mixed_traffic_model_specific_test_ratios   
5   exp4_mixed_traffic_model_specific_test_ratios   
6   exp4_mixed_traffic_model_specific_test_ratios   
7   exp4_mixed_traffic_model_specific_test_ratios   
8   exp4_mixed_traffic_model_specific_test_ratios   
9   exp4_mixed_traffic_model_specific_test_ratios   
10  exp4_mixed_traffic_model_specific_test_ratios   
11  exp4_mixed_traffic_model_specific_test_ratios   
12  exp4_mixed_traffic_model_specific_test_ratios   
13  exp4_mixed_traffic_model_specific_test_ratios   
14  exp4_mixed_traffic_model_specific_test_ratios   
15  exp4_mixed_traffic_model_specific_test_ratios   
16  exp4_mixed_traffic_model_specific_test_ratios   
17  exp4_mixed_traffic_model_specific_test_ratios   
18  exp4_mixed_traffic_model_specific_test_ratios   
19  exp4_mixed_traffic_model_specific_test_ratios   
20  exp4_mixed_traffic_model_specific_test_ratios   
21  exp4_mixed_traffic_model_specific_test_ratios   
22  exp4_mixed_traffic_model_specific_test_ratios   
23  exp4_mixed_traffic_model_specific_test_ratios   
24  exp4_mixed_traffic_model_specific_test_ratios   
25  exp4_mixed_traffic_model_specific_test_ratios   
26  exp4_mixed_traffic_model_specific_test_ratios   
27  exp4_mixed_traffic_model_specific_test_ratios   
28  exp4_mixed_traffic_model_specific_test_ratios   
29  exp4_mixed_traffic_model_specific_test_ratios   

                         training_variant                  model  \
                                                                   
0   balanced_mixed_train_model_ratio_test              buhl_dcnn   
1   balanced_mixed_train_model_ratio_test              buhl_dcnn   
2   balanced_mixed_train_model_ratio_test              buhl_dcnn   
3   balanced_mixed_train_model_ratio_test              buhl_dcnn   
4   balanced_mixed_train_model_ratio_test              buhl_dcnn   
5   balanced_mixed_train_model_ratio_test              buhl_dcnn   
6   balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
7   balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
8   balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
9   balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
10  balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
11  balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
12  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
13  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
14  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
15  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
16  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
17  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
18  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
19  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
20  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
21  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
22  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
23  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
24  balanced_mixed_train_model_ratio_test      zkr_random_forest   
25  balanced_mixed_train_model_ratio_test      zkr_random_forest   
26  balanced_mixed_train_model_ratio_test      zkr_random_forest   
27  balanced_mixed_train_model_ratio_test      zkr_random_forest   
28  balanced_mixed_train_model_ratio_test      zkr_random_forest   
29  balanced_mixed_train_model_ratio_test      zkr_random_forest   

           test_source  accuracy                                \
                            mean       std       min       max   
0          CIC-IDS2017  0.997195 

experiment  \
                                                   
0  exp4_mixed_traffic_model_specific_test_ratios   
1  exp4_mixed_traffic_model_specific_test_ratios   
2  exp4_mixed_traffic_model_specific_test_ratios   
3  exp4_mixed_traffic_model_specific_test_ratios   
4  exp4_mixed_traffic_model_specific_test_ratios   

                        training_variant                  model  \
                                                                  
0  balanced_mixed_train_model_ratio_test              buhl_dcnn   
1  balanced_mixed_train_model_ratio_test       elmasry_dpso_dnn   
2  balanced_mixed_train_model_ratio_test  kamal_transformer_dnn   
3  balanced_mixed_train_model_ratio_test         kilichev_1dcnn   
4  balanced_mixed_train_model_ratio_test      zkr_random_forest   

  source_macro_f1                                \
             mean       std       min       max   
0        0.771070  0.028380  0.733935  0.810940   
1        0.887810  0.049625  0.821988  0.945847   
2        0.782032  0.019040  0.760116  0.805823   
3        0.765571  0.004254  0.759408  0.770753   
4        0.699291  0.023143  0.659426  0.716863   

  source_macro_balanced_accuracy                                \
                            mean       std       min       max   
0                       0.865863  0.012269  0.850802  0.884287   
1                       0.832317  0.018091  0.801431  0.847532   
2                       0.855798  0.006349  0.848367  0.863767   
3                       0.867276  0.001755  0.864790  0.869170   
4                       0.814885  0.010740  0.799095  0.828860   

  source_macro_recall                               source_macro_specificity  \
                 mean       std       min       max                     mean   
0            0.767422  0.025159  0.736253  0.805902                 0.964304   
1            0.834210  0.077068  0.743551  0.931191                 0.830423   
2            0.747086  0.013306  0.732612  0.765092                 0.964509   
3            0.772066  0.002867  0.768056  0.775611                 0.962487   
4            0.673106  0.026093  0.640356  0.712177                 0.956665   

                                 
        std       min       max  
0  0.002001  0.962447  0.967232  
1  0.050172  0.763873  0.872984  
2  0.003311  0.961480  0.968442  
3  0.000791  0.961523  0.963464  
4  0.006837  0.945544  0.962377

,experiment,training_variant,model,test_source,test_category,support,attack_recall_mean,attack_recall_std,benign_specificity_mean,benign_specificity_std
0,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017,benign,145100,NaN,NaN,0.996572,0.000479
1,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017,dos,35554,0.999764,0.000101,NaN,NaN
2,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017,web_attack,1,0.000000,0.000000,NaN,NaN
3,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017-native,benign,414321,NaN,NaN,0.975284,0.002929
4,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,buhl_dcnn,CIC-IDS2017-native,botnet,691,0.001447,0.001023,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
125,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,zkr_random_forest,UNSW-NB15,fuzzers,879,0.817747,0.062980,NaN,NaN
126,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,zkr_random_forest,UNSW-NB15,generic,183,0.849180,0.053065,NaN,NaN
127,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,zkr_random_forest,UNSW-NB15,reconnaissance,405,0.949136,0.030472,NaN,NaN
128,exp4_mixed_traffic_model_specific_test_ratios,balanced_mixed_train_model_ratio_test,zkr_random_forest,UNSW-NB15,shellcode,61,0.875410,0.078278,NaN,NaN
